## Structured Output

Models can be requested to provide their response in a format that matches a given schema. This is useful for ensuring that the output can be easily parsed and used in subsequent processing.

LangChain supports multiple schema types and methods for enforcing structured output.

### Benefits

- Ensures consistent response formats
- Simplifies parsing and validation
- Reduces post-processing effort
- Improves reliability in production workflows

### Common Use Cases

- API response generation
- Data extraction
- Form validation
- Agent-to-agent communication
- Database record creation

### Pydantic 

Pydantic models provides the richest feature with field validation, descriptions, and nested structures.

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The Title of the Movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

In [ ]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

In [ ]:
# Generic response given by AI
model.invoke("provide details of the movie Inception")

AIMessage(content='**Inception (2010)**\n\nInception is a science fiction action film written, co-produced, and directed by Christopher Nolan. The movie stars Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, and Marion Cotillard.\n\n**Plot**\n\nThe movie follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts, including Arthur (Joseph Gordon-Levitt), a point man; Aria

In [ ]:
# Structured response which we formatted using Pydantic
response = model_with_structure.invoke("provide details of the movie Inception")
response 

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.5)

### Nested Structure 

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Action', 'Sci-Fi'], budget=160.0)

### TypedDict

In [2]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movire with details."""
    title: Annotated[str,...,"The Title of the Movie"]
    year: Annotated[str,...,"This year the movie was released"]
    director: Annotated[str,...,"The director of the movie"]
    rating: Annotated[str,...,"The movie's rating out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
response = model_withtypeddict.invoke("Please provide me the details of the Movie Avengers EndGame")
response

{'director': 'Anthony Russo, Joseph Russo',
 'rating': '8.4',
 'title': 'Avengers EndGame',
 'year': '2019'}

### DataClasses

In [3]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model,
    response_format=ContactInfo    # Auto-selects ProviderStratergy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [ ]:
## DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact info for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model,
    response_format=ContactInfo
)

result = agent.invoke({
        "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]

})
result

InvalidUpdateError: Expected dict, got Extract contact info from: John Doe, john@example.com, (555) 123-4567
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE